In [1]:
# ============================================================
# NLP + K-MEANS FOR test_coffee_output.csv
# ============================================================

# Install if needed:
# !pip install pandas scikit-learn nltk matplotlib

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Download NLP resources
nltk.download("stopwords")
nltk.download("wordnet")

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv("test_coffee_output.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. FIND TEXT COLUMN
# ------------------------------------------------------------

possible_columns = [
    "review", "reviews", "text", "comment",
    "comments", "description", "feedback", "content"
]

text_col = next(
    (col for col in possible_columns if col in df.columns),
    None
)

# If no common text column exists, use first string column
if text_col is None:
    string_cols = df.select_dtypes(include="object").columns
    if len(string_cols) > 0:
        text_col = string_cols[0]
    else:
        raise ValueError("No text column found.")

print("Using text column:", text_col)

# ------------------------------------------------------------
# 3. CLEAN TEXT - NLP
# ------------------------------------------------------------

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()

    words = [
        lemmatizer.lemmatize(w)
        for w in words
        if w not in stop_words and len(w) > 2
    ]

    return " ".join(words)

df[text_col] = df[text_col].fillna("")
df["clean_text"] = df[text_col].apply(clean_text)

# Remove empty rows
df = df[df["clean_text"].str.strip() != ""].copy()

# ------------------------------------------------------------
# 4. TF-IDF
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2
)

X = tfidf.fit_transform(df["clean_text"])

print("TF-IDF shape:", X.shape)

# ------------------------------------------------------------
# 5. FIND BEST K
# ------------------------------------------------------------

scores = {}

for k in range(2, 8):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)

print("\nSilhouette scores:")
for k, score in scores.items():
    print(f"K={k}: {score:.3f}")

print("\nSelected K:", best_k)

# ------------------------------------------------------------
# 6. K-MEANS
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

# ------------------------------------------------------------
# 7. TOP WORDS IN EACH CLUSTER
# ------------------------------------------------------------

terms = tfidf.get_feature_names_out()

print("\n" + "="*60)
print("TOP WORDS PER CLUSTER")
print("="*60)

for cluster in range(best_k):

    centroid = kmeans.cluster_centers_[cluster]

    top_indices = centroid.argsort()[-15:][::-1]

    words = [terms[i] for i in top_indices]

    print(f"\nCluster {cluster}:")
    print(", ".join(words))

# ------------------------------------------------------------
# 8. CLUSTER COUNTS
# ------------------------------------------------------------

print("\n" + "="*60)
print("CLUSTER COUNTS")
print("="*60)

print(df["cluster"].value_counts().sort_index())

# ------------------------------------------------------------
# 9. SHOW SAMPLE REVIEWS FROM EACH CLUSTER
# ------------------------------------------------------------

for cluster in range(best_k):

    print("\n" + "="*60)
    print(f"CLUSTER {cluster} EXAMPLES")
    print("="*60)

    display(
        df[df["cluster"] == cluster][
            [text_col, "cluster"]
        ].head(5)
    )

# ------------------------------------------------------------
# 10. VISUALISE CLUSTERS
# ------------------------------------------------------------

pca = PCA(n_components=2, random_state=42)

X_2D = pca.fit_transform(X.toarray())

plt.figure(figsize=(10, 6))

plt.scatter(
    X_2D[:, 0],
    X_2D[:, 1],
    c=df["cluster"],
    cmap="viridis",
    alpha=0.7
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Coffee Reviews - NLP + K-Means Clustering")
plt.colorbar(label="Cluster")
plt.show()

# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

df.to_csv(
    "coffee_kmeans_nlp_results.csv",
    index=False
)

print("\nDONE!")
print("Output file: coffee_kmeans_nlp_results.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_6431/346958542.py:49: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include="object").columns


Dataset shape: (6, 5)
Columns: ['index', 'product_name_en', 'coicop_1', 'coicop_2', 'coicop_3']
Using text column: product_name_en
TF-IDF shape: (6, 1)


/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (6). Possibl

ValueError: n_samples=6 should be >= n_clusters=7.